# Transmission System **with Impairments** (4-PAM)

*Python/Colab conversion of the MATLAB script `impsys.m`*
(from **Software Receiver Design**, Johnson, Sethares & Klein).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/CommSystems_Course/blob/main/05_impsys.ipynb)

> **How to run:** Click the **Open in Colab** badge above, then run every cell top-to-bottom with `Shift + Enter`, or use **Runtime -> Run all**. No login needed — just click and run.

This is the **real-world** version of the ideal system (`idsys`). It sends the same text message, but now through a channel with **uncompensated impairments**:

- **Channel noise** (Gaussian)
- **Multipath / ISI** (echoes of the signal)
- **Carrier frequency & phase offset** (transmitter vs. receiver oscillator mismatch)
- **Timing offsets** (baud timing and symbol-period error)

You'll see how each impairment degrades the **eye diagram**, the **soft-decision clusters**, the **cluster variance**, and ultimately the **symbol-error rate**.

No installation needed — `numpy`, `scipy`, `matplotlib` and `plotly` are pre-installed in Colab.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import remez, lfilter          # remez = MATLAB firpm; lfilter = MATLAB filter
from scipy.signal.windows import hamming          # MATLAB hamming(M)

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'])
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 4)

## 2. Textbook helper functions (re-implemented in Python)

Same toolbox helpers as the ideal system. **`quantalph`** is ported directly from the MATLAB source you provided — it quantizes each input value to the nearest alphabet symbol using the **squared-distance (nearest-neighbor)** method.

| MATLAB helper | Purpose |
|---|---|
| `letters2pam` / `pam2letters` | text ↔ 4-PAM symbols {−3,−1,1,3} |
| `quantalph` | nearest-neighbor quantization to the alphabet |
| `pow` | signal power = mean of squares |
| `plotspec` | waveform + magnitude spectrum |

In [ ]:
def letters2pam(s):
    """Encode ASCII string as a 4-PAM sequence in {-3,-1,1,3} (4 symbols/char)."""
    x = []
    for ch in s:
        v = ord(ch)
        digits = [(v // 64) % 4, (v // 16) % 4, (v // 4) % 4, v % 4]
        x += [2 * d - 3 for d in digits]
    return np.array(x, dtype=float)


def pam2letters(seq):
    """Decode a 4-PAM sequence back into an ASCII string."""
    seq = np.asarray(seq); out = []
    n = (len(seq) // 4) * 4
    for k in range(0, n, 4):
        d = [min(max(int(round((s + 3) / 2)), 0), 3) for s in seq[k:k+4]]
        out.append(chr((d[0]*64 + d[1]*16 + d[2]*4 + d[3]) % 256))
    return ''.join(out)


def quantalph(x, alphabet):
    """Quantize x to the alphabet using nearest neighbor (port of quantalph.m).

    MATLAB:
        dist=(x - alpha).^2;  [v,i]=min(dist,[],2);  y=alphabet(i);
    """
    x = np.asarray(x).reshape(-1, 1)
    a = np.asarray(alphabet).reshape(1, -1)
    dist = (x - a) ** 2                     # squared distance to each symbol
    i = np.argmin(dist, axis=1)            # index of nearest symbol
    return np.asarray(alphabet).ravel()[i]


def pow_(x):
    """Signal power: mean of squares (MATLAB pow.m)."""
    x = np.asarray(x)
    return np.sum(x**2) / len(x)


def plotspec(x, Ts, flim=None, title=''):
    """Waveform + magnitude spectrum (Python port of plotspec.m), optional zoom."""
    x = np.asarray(x); N = len(x)
    t = Ts * np.arange(1, N + 1)
    ssf = np.arange(-N/2, N/2) / (Ts * N)
    fxs = np.fft.fftshift(np.fft.fft(x))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6))
    ax1.plot(t, x); ax1.set_xlabel('seconds'); ax1.set_ylabel('amplitude')
    ax1.set_title(title or 'Waveform'); ax1.grid(True)
    ax2.plot(ssf, np.abs(fxs)); ax2.set_xlabel('frequency (Hz)')
    ax2.set_ylabel('magnitude'); ax2.set_title('Magnitude spectrum'); ax2.grid(True)
    if flim is not None:
        lo, hi = (-flim, flim) if np.isscalar(flim) else (flim[0], flim[1])
        ax2.set_xlim(lo, hi)
    fig.tight_layout(); plt.show()

## 3. Choose the impairments  🎛️

In the notebook, just **edit the variables below and re-run** (Runtime -> Run all). Start with all zeros for the ideal case, then turn impairments on one at a time to see their effect.

| variable | meaning | try |
|---|---|---|
| `cng`   | channel noise gain | `0`, `0.6`, `2` |
| `cdi`   | channel multipath | `0` none, `1` mild, `2` harsh |
| `fo`    | transmitter mixer **frequency** offset (%) | `0`, `0.01` |
| `po`    | transmitter mixer **phase** offset (rad) | `0`, `0.7`, `0.9` |
| `toper` | baud **timing** offset (% of symbol period) | `0`, `20`, `30` |
| `so`    | **symbol-period** offset | `0`, `1` |

In [ ]:
# ---- EDIT THESE to explore different impairments ----
cng   = 0      # channel noise gain:            try 0, 0.6 or 2
cdi   = 0      # channel multipath:             0 none, 1 mild, 2 harsh
fo    = 0      # tx mixer freq offset (percent): try 0 or 0.01
po    = 0      # tx mixer phase offset (rad):    try 0, 0.7 or 0.9
toper = 0      # baud timing offset (percent):   try 0, 20 or 30
so    = 0      # symbol period offset:           try 0 or 1
# ------------------------------------------------------

# Reproducibility for the random channel noise (comment out for fresh noise each run)
np.random.seed(0)

## 4. Transmitter

Text → 4-PAM symbols → upsample → Hamming pulse shaping → modulate onto the carrier. The carrier now includes the **frequency (`fo`) and phase (`po`) offsets** relative to the receiver's oscillator.

In [ ]:
# encode text as a T-spaced 4-PAM sequence
s = '01234 I wish I were an Oscar Mayer wiener 56789'
m = letters2pam(s); N = len(m)          # 4-level signal of length N

M = 100 - so                            # oversampling factor (>= 8)
mup = np.zeros(N * M); mup[::M] = m     # upsample: one symbol every M samples

p = hamming(M)                          # Hamming 'blip' pulse of width M
x = lfilter(p, 1, mup)                 # pulse-shape the data

t = np.arange(1, len(x) + 1) / M        # T/M-spaced time vector
fc = 20                                 # carrier frequency
c = np.cos(2*np.pi*(fc*(1 + 0.01*fo))*t + po)   # carrier WITH freq & phase offset
r = c * x                               # modulate message with carrier

print(f'message: {N} symbols, M = {M} samples/symbol, waveform: {len(x)} samples')

**Baseband signal spectrum** :

In [ ]:
plotspec(x, 1/M, title='Baseband pulse-shaped signal x')

## 5. Channel  — multipath + noise + timing

The **multipath channel** `mc` adds delayed echoes (inter-symbol interference), is power-normalized, then Gaussian **noise** of gain `cng` is added. Finally a fractional **timing offset** `to` shifts where symbols are sampled.

In [ ]:
# --- channel inter-symbol interference (ISI) ---
if cdi < 0.5:
    mc = np.array([1.0, 0.0, 0.0])                       # distortion-free channel
elif cdi < 1.5:
    mc = np.concatenate([[1.0], np.zeros(M), [0.28],
                         np.zeros(int(2.3*M)), [0.11]])  # mild multipath
else:
    mc = np.concatenate([[1.0], np.zeros(M), [0.28],
                         np.zeros(int(1.8*M)), [0.44]])  # harsh multipath

mc = mc / np.sqrt(np.sum(mc * mc))          # normalize channel power
dv = lfilter(mc, 1, r)                       # filter signal through channel
nv = dv + cng * np.random.randn(len(dv))     # add Gaussian channel noise

to = int(np.floor(0.01 * toper * M))         # fractional-period delay (samples)
rnv = nv[to:N*M]                             # MATLAB nv(1+to:N*M) -> [to:N*M]
rt = np.arange(1 + to, len(nv) + 1) / M      # time vector with delayed start
rt = rt[:len(rnv)]
rM = M + so                                  # receiver sampler timing offset

print(f'channel taps = {len(mc)}, timing delay to = {to} samples, rM = {rM}')

## 6. Receiver

Demodulate with a synchronized cosine, low-pass filter (`firpm`→`remez`), then matched-filter with the receiver's pulse shape. Note the receiver uses **`rM`** samples/symbol — which differs from the transmitter's `M` when `so ≠ 0`, modeling a sample-rate mismatch.

In [ ]:
c2 = np.cos(2*np.pi*fc*rt)               # synchronized cosine for mixing
x2 = rnv * c2                            # demodulate received signal

fl = 50                                  # LPF length
b = remez(fl + 1, [0, 0.1, 0.2, 1], [1, 0], fs=2)   # LPF (firpm equivalent)
x3 = 2 * lfilter(b, 1, x2)               # LPF and scale downconverted signal

rp = hamming(rM)                         # receiver-defined pulse shape
y = lfilter(np.flip(rp) / (pow_(rp) * rM), 1, x3)   # matched filter
print('receiver output y computed:', len(y), 'samples')

## 7. Eye diagram 

The **eye diagram** overlays many short segments of the received signal (4 symbol periods wide). A wide-open 'eye' means symbols are easy to distinguish; impairments **close the eye**, making errors more likely. This is the single most useful diagnostic in digital receiver design.

In [ ]:
ul = int(np.floor((len(y) - 124) / (4 * rM)))     # number of overlaid traces
seg = y[124:ul*4*rM + 124]                          # MATLAB y(125 : ul*4*rM+124)
eye = seg.reshape(ul, 4*rM).T                       # column-major reshape (4*rM x ul)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(eye, 'b', linewidth=0.5, alpha=0.6)
ax.set_xlabel('samples (4 symbol periods wide)')
ax.set_ylabel('amplitude')
ax.set_title('Eye diagram  (wider \u201copen\u201d eyes = easier decisions)')
ax.grid(True)
fig.tight_layout(); plt.show()

## 8. Downsample, decide, and assess performance

In [ ]:
# downsample: MATLAB y(0.5*fl+rM : rM+so : N*M-to) is 1-indexed
start = int(0.5*fl + rM)
stop  = N*M - to
step  = rM + so
idx = np.arange(start, stop + 1, step) - 1     # to 0-indexed
idx = idx[idx < len(y)]
z = y[idx]                                      # soft decisions (one per symbol)

mprime = quantalph(z, [-3, -1, 1, 3])          # hard decisions
lmp = len(mprime)

cluster_variance = np.sum((mprime - z)**2) / lmp
percentage_symbol_errors = 100 * np.sum(np.abs(np.sign(mprime - m[:lmp]))) / lmp

print(f'cluster variance          : {cluster_variance:.5f}')
print(f'percentage symbol errors  : {percentage_symbol_errors:.2f} %')
print()
reconstructed_message = pam2letters(mprime)
print('original     :', repr(s))
print('reconstructed:', repr(reconstructed_message))

**Soft decisions**  — tightly clustered on ±1, ±3 = healthy link:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.arange(1, len(z) + 1), z, '.')
for lvl in [-3, -1, 1, 3]:
    ax.axhline(lvl, color='r', ls='--', lw=0.7, alpha=0.6)
ax.set_xlabel('symbol index'); ax.set_ylabel('soft decision value')
ax.set_title('Soft decisions'); ax.grid(True)
fig.tight_layout(); plt.show()

## 9. Notes & things to try

- **Compare to `idsys`:** with all impairments at 0 you get the ideal system back — tight clusters, a wide-open eye, and 0% symbol error. Turn impairments on to watch everything degrade.
- **One at a time:** the cleanest way to build intuition is to change a *single* impairment, re-run (**Runtime -> Run all**), and watch the **eye diagram** and **cluster variance** respond.
- **What each impairment does:**
  - `cng` (noise) → fuzzes the clusters and eye traces.
  - `cdi` (multipath) → inter-symbol interference smears symbols together.
  - `fo`/`po` (carrier offset) → rotates/scales the whole constellation (phase offset `po=0.9` is especially damaging — the receiver isn't correcting it).
  - `toper`/`so` (timing) → samples the eye away from its widest point.
- **firpm/filter mapping:** `firpm` → `scipy.signal.remez(fl+1, bands, desired, fs=2)`; `filter(b,1,x)` → `lfilter(b,1,x)`; MATLAB 1-indexing → subtract 1 in Python.
- **Fresh noise each run:** comment out `np.random.seed(0)` in the impairments cell.
